# Introduction

When representing binary relationships, such as familial relationships, a number of approaches may be utilized including: ad-hoc, general graphs, specialized tables or charts. As I've explored different representations, a specialized mathematical representation has emerged. This mathematical representation is useful as the underpinnings of dynamic graph algorithms for use cases that include graph analysis, relationship calculation, loop detection, loop prevention, etc. 

The underlying mathematical model is a directed, acyclic graph of multiple, interleaved binary trees, designated as a Red Black Graph. The name, Red Black Graph, derives from superficial similarity to Red Black Trees. Red Black Trees are binary trees such that each node has an extra, color bit (red or black). This color bit is used to balance the tree as modifications are made. In a Red Black Graph each vertex also has a an extra, color bit, rather than utilizing the color bit for balancing, the color bit is used to constrain edges between vertices.

I will provide a formal definition of a Red Black Graph, as well as explore a number of interesting emergent properties. I will also examine several applications of Red Black Graphs to illustrate the utility of using this mathematical model for familial relationships.

## Formal Definition

Let $C = \{\mathsf r, \mathsf b\}$ be two colors, red and black, identified with the bits $\mathsf r = 0$ and $\mathsf b = 1$.

A **Red Black Graph** is a finite directed graph, $G = (V,E)$, together with a coloring $c : V \rightarrow C$, such that:

1. **(Acyclic)** $G$ has no directed cycles. In particular, $E$ has no self-loops.
2. **(One edge per color)** For every vertex $\mathbf{u}$ and every color $k$, there is at most one edge $(\mathbf{u}, \mathbf{v}) \in E$ with $c(\mathbf{v}) = k$.

The *word* of a directed path $p = (\mathbf{v}_0 \to \mathbf{v}_1 \to \dots \to \mathbf{v}_n)$ is the sequence of colors of the vertices it enters, $w(p) = c(\mathbf{v}_1) \cdots c(\mathbf{v}_n)$. The empty path has the empty word, $\varepsilon$.

Words are encoded as positive integers by $\iota(\varepsilon) = 1$ and $\iota(wk) = 2\,\iota(w) + k$: a leading $1$ bit followed by the bits of the word. Words are ordered *shortlex*: shorter words first, and words of equal length lexicographically, with $\mathsf r$ before $\mathsf b$.

The **relationship function** is

$$r(\mathbf{u},\mathbf{v}) = \begin{cases}
\iota\big(\min_{\text{shortlex}} \{\, w(p) : p \text{ is a path from } \mathbf{u} \text{ to } \mathbf{v} \,\}\big), & \text{if such a path exists,} \\
0, & \text{otherwise.}
\end{cases}$$

Put procedurally: walk the shortest path from $\mathbf{u}$ to $\mathbf{v}$, breaking ties in favor of red. Start with $x \leftarrow 1$, and on entering each vertex update
$$x \leftarrow \begin{cases}
2x, & \text{if the vertex is red}, \\
2x + 1, & \text{if the vertex is black}.
\end{cases}$$
In particular $r(\mathbf{u},\mathbf{u}) = 1$, the empty path.

When relationships are stored in a matrix, the diagonal is also used to record each vertex's color: $-1$ for a red vertex and $1$ for a black vertex. This is a display convention; it does not change any other entry (see "The ±1 Diagonal" below).

## Motivation

The relationships resulting from sexual reproduction can be modeled by a Red Black Graph, arbitrarily assigning vertices that are male as Red and vertices that are female as Black with direction of edges being from the offspring to the parent.  

<!-- to generate latex using nbconvert use the image markdown, for display use the inline html --> 
<img src="img/simple-graph.png" alt="Sample Graph" style="width: 300px;"/>
<!-- ![Simple Red Black Graph](img/simple-graph.png){ width=50% } -->

## Observation
For a given vertex in a Red Black graph there are two distinct sub-graphs or "views" or perspectives, for a given vertex: Descendency and Ancestry. 

**Descendency** is the sub-graph for a given vertex that consists of all vertices and edges that can follow a graph traversal and arrive at the given node.

<img src="img/simple-graph-descendency.png" alt="Descendency View" style="width: 300px;"/>
<!-- ![Descendency View for Female Progenitor](img/simple-graph-descendency.png){ width=50% } -->

**Ancestry** is the sub-graph for a given vertex that consists of all the vertices and edges reachable by following out-bound edges.

<img src="img/simple-graph-ancestry.png" alt="Ancestry View" style="width: 300px;"/>
<!-- ![Ancestry View for Female Offspring](img/simple-graph-ancestry.png){ width=50% } -->

Because each vertex has at most one out-bound edge per color, a word determines at most one path from a given vertex. The *unfolding* of the ancestry, with one node for each word that can be walked, is therefore a binary tree. The ancestry sub-graph itself is a binary tree only when no ancestor can be reached by two different words. Pedigree collapse, for example descent from the same ancestor through both parents, makes it a DAG.

$r(\mathbf{u},\mathbf{v})$ numbers the nodes of the unfolded tree in breadth first order. This is the Ahnentafel numbering (also called Sosa–Stradonitz or Eytzinger numbering) used in pedigree charts in family history research: self is 1, father 2, mother 3, and in general the father of $n$ is $2n$ and the mother is $2n+1$. When an ancestor appears at several positions in the chart, $r$ is the smallest of its numbers. See fig. "Pedigree Chart".

<img src="img/pedigree-1.png" alt="Pedigree Chart" style="width: 800px;"/>
<!-- ![Pedigree Chart](img/pedigree-1.png){ width=80% } -->



## Adjacency Matrix

An adjacency matrix is a square matrix used to represent a graph. The elements of the matrix are 1 (or edge weight) if there is an edge between the vertices represented by the column index and the row index. Slightly more formally, for $G$ with a vertex set $V$, the adjacency matrix is a square $|V|$ x $|V|$ matrix, $A$, such that $A_{ij}$ is $1$ if there exists an edge from $\mathbf{i}$ to $\mathbf{j}$ and $0$ otherwise.

Given the above example graph and chosing indices for the vertices as follows: 0 - Female Offspring, 1 - Male Offspring, 2 - Male Progenitor, 3 - Female Progenitor, the graph would be represented by the following adjacency matrix.

$$A = \begin{bmatrix}
0 & 0 & 1 & 1 \\
0 & 0 & 1 & 1 \\
0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 \\
\end{bmatrix}$$

## Red Black Graph Adjacency Matrix

I'll define the adjacency matrix for a Red Black Graph only slightly differently. $R_{ij}=r(\mathbf{i}, \mathbf{j})$ if there is an edge from $\mathbf{i}$ to $\mathbf{j}$ and $0$ otherwise. The following is the Red Black adjacency matrix for the above example graph.

$$R = \begin{bmatrix}
1 & 0 & 2 & 3 \\
0 & -1 & 2 & 3 \\
0 & 0 & -1 & 0 \\
0 & 0 & 0 & 1 \\
\end{bmatrix}$$

Observe the following properties:

$$trace(R) = |V_{black}| - |V_{red}|$$

$$|V| = |V_{black}| + |V_{red}|$$

$$|V_{black}| = \frac{|V| + trace(R)}{2}$$

$$|V_{red}| = \frac{|V| - trace(R)}{2}$$

These identities use ordinary integer arithmetic on the diagonal. They are not avos operations.

## Transitive Closure

Computing the transitive closure of an adjacency matrix, $A$, results in the reachability a matrix, $A^+$, that shows all vertices that are reachable from any given vertex. If $A_{ij} == 1$ there is a path from $\mathbf{v}_i$ to $\mathbf{v}_j$.

The transitive closure of a Red Black adjacency matrix, $R$, is defined to be the relationship matrix, $R^+$. In addition to reachability, $R^+$ is defined in such that if $R^+_{ij} == \mathbf{n}$ and $\mathbf{n}$ is non-zero, then from $\mathbf{n}$ we can derive both the path lengh and the explicit traversal path from $\mathbf{i}$ to $\mathbf{j}$.

As an example consider the following graph, where each node has been labeled with a vertex index:

<img src="img/simple-graph-transitive-closure.png" alt="Graph for Transitive Closure" style="width: 100px;"/>
<!-- ![Red Black Graph Example for Transitive Closure](img/simple-graph-transitive-closure.png){ width=20% } -->

By inspection:

$$R = \begin{bmatrix}
-1 & 2 & 3 & 0 & 0 \\
0 & -1 & 0 & 2 & 0 \\
0 & 0 & 1 & 0 & 0 \\
0 & 0 & 0 & -1 & 0 \\
2 & 0 & 0 & 0 & 1 \\
\end{bmatrix}\text{ and }R^+ = \begin{bmatrix}
-1 & 2 & 3 & 4 & 0 \\
0 & -1 & 0 & 2 & 0 \\
0 & 0 & 1 & 0 & 0 \\
0 & 0 & 0 & -1 & 0 \\
2 & 4 & 5 & 8 & 1 \\
\end{bmatrix}$$

Before examining how to generate $R^+$ from $R$, consider the following observations:

* For $\mathbf{u} \neq \mathbf{v}$, $r(\mathbf{u},\mathbf{v})$ is even when $\mathbf{v}$ is red and odd when $\mathbf{v}$ is black: its last bit is the color of the last vertex on the path (Theorem 3).
* The path length is inherent in $r(\mathbf{u},\mathbf{v})$. It is the position of the most significant bit, the integral portion of $log_{2}(r(\mathbf{u},\mathbf{v}))$.
* The traversal path is also inherent in $r(\mathbf{u},\mathbf{v})$. The bits after the leading 1, read from most to least significant, give the color of each vertex entered along the path. Since a vertex has at most one parent of each color, the colors determine the path.
* The diameter (the longest shortest path) is the integral portion of $log_{2}(max(r(\mathbf{u},\mathbf{v})))$.

## Red Black Arithmetic

As constructing $R^+$ by inspection is cumbersome for non-trivial cases, I'll explain a method to algorithmically derive $R^+$ from $R$. 

### Avos Product
The first step requires defining a transitive relationship function. To illustrate, consider 3 vertices: $\mathbf{u}$, $\mathbf{v}$ and $\mathbf{w}$. Further, assume that there is a path from $\mathbf{u}$ to $\mathbf{v}$ and from $\mathbf{v}$ to $\mathbf{w}$. I'll define the function, $f(r(\mathbf{u},\mathbf{v}),r(\mathbf{v},\mathbf{w}))\rightarrow r(\mathbf{u},\mathbf{w})$, as the *avos product* and designate it using $\star$, e.g. $r(\mathbf{u},\mathbf{w})=r(\mathbf{u},\mathbf{v}) \star r(\mathbf{v},\mathbf{w})$.

If $r(\mathbf{u},\mathbf{v})=4$ ($\mathbf{v}$ is $\mathbf{u}$'s paternal grandfather) and $r(\mathbf{v},\mathbf{w})=7$ ($\mathbf{w}$ is $\mathbf{v}$'s maternal grandmother). Using either the defintion of $r$ or transcribing $\mathbf{v}$'s pedigree into the proper place in $\mathbf{u}$'s pedigree chart we see that $r(\mathbf{u},\mathbf{w})=19$, or $19 = 4 \star 7$. To further explore this consider the following ($r(\mathbf{x},\mathbf{y})$ represented in base-2 for illustrative purposes):

| $\mathbf{v}$'s relationship to $\mathbf{u}$  | $\mathbf{w}$'s relationship to $\mathbf{v}$ | $r(\mathbf{u},\mathbf{v})$ | $r(\mathbf{v},\mathbf{w})$ | $r(\mathbf{u},\mathbf{w})$ |
| ---------------------- | --------------------- | --------- | -------- | --------- |
| father                 | father                | 10        | 10       | 100       |
| father                 | mother                | 10        | 11       | 101       |
| mother                 | father                | 11        | 10       | 110       |
| mother                 | mother                | 11        | 11       | 111       |
| father                 | paternal grandfather  | 10        | 100      | 1000      |
| maternal grandmother   | paternal grandfather  | 111 &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; | 100 &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; | 11100 &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; |

While perhaps not obvious, upon examination of the binary representation, the avos product simply replaces the most significant bit of $r(\mathbf{v},\mathbf{w})$ with the value of $r(\mathbf{u},\mathbf{v})$. 

In terms of words, the avos product is concatenation: the path from $\mathbf{u}$ to $\mathbf{v}$ followed by the path from $\mathbf{v}$ to $\mathbf{w}$. With $m = \operatorname{MSB}(y)$, the position of the most significant bit of $y$,

$$x \star y = (x \ll m) \mid (y - 2^m),$$

and $x \star 0 = 0 \star y = 0$. The value $1$, the empty path, is an identity on both sides: $1 \star y = y$ and $x \star 1 = x$.

### Avos Sum
It is natural to pair multiplication with addition for linear algebra operations (matrix multiplication, matrix distance product, etc.). The *avos sum*, designated by $\dagger$, is the minimum, with the convention that $0$ (no path) is larger than every other value, e.g. $2 = 37 \dagger 2$ and $3 = 0 \dagger 3$. Because integer order is the same as shortlex order on words (Lemma 2 below), the avos sum of two relationships selects the shorter path, and between paths of the same length, the one that comes first with red before black.

**Storage note.** As, aside from the diagonal, the range of $r(\mathbf{u},\mathbf{v})$ is $\mathbb{N}$, it is natural to use unsigned numeric types in a strongly typed language. The diagonal value $-1$ is then stored as $2^\mathbf{w}-1$, where $\mathbf{w}$ is the word size. Signed and unsigned storage order $-1$ differently under the minimum, but on a valid graph a diagonal entry is never compared with another non-zero value, so both give the same results.

The reference implementation below also gives the product special rules for $-1$ and $1$ on the diagonal. These are explained in "The ±1 Diagonal", after the foundations.

### Reference Implementation of Avos Arithmetic

In [ ]:
# %load ../redblackgraph/reference/rbg_math.py
def MSB(x: int) -> int:
    '''
    Given an integer, return the bit position of it's most significant bit
    :param x: operand
    :return: bit position of leftmost significant bit
    '''
    bit_position = 0
    while (x > 1):
        x >>= 1
        bit_position += 1
    return bit_position

def avos_sum(x: int, y: int) -> int:
    '''
    The avos sum is the non-zero minimum of x and y
    :param x: operand 1
    :param y: operand 2
    :return: avos sum
    '''
    if x == 0:
        return y
    if y == 0:
        return x
    if x < y:
        return x
    return y

def avos_product(x: int, y: int) -> int:
    '''
    The avos product operation in the red-black semiring.
    
    Parity identity constraints:
    - RED_ONE (-1) is identity for even values, annihilator for odd values
    - BLACK_ONE (1) is identity for odd values, annihilator for even values
    
    Rules:
    - even ⊗ RED_ONE = even (identity)
    - odd ⊗ RED_ONE = 0 (annihilator)
    - odd ⊗ BLACK_ONE = odd (identity)
    - even ⊗ BLACK_ONE = 0 (annihilator)
    
    :param x: operand 1
    :param y: operand 2
    :return: avos product
    '''

    # negative values are invalid (aside from -1)
    if x < -1 or y < -1:
        raise ValueError(f"Invalid input. Negative values (aside from -1) are not allowed. x: {x}, y:{y}")
    
    # The zero property of the avos product
    if x == 0 or y == 0:
        return 0
    
    # Identity ⊗ Identity special cases (must come before other checks)
    # Same-gender self-loops
    if x == -1 and y == -1:
        return -1  # RED_ONE ⊗ RED_ONE = RED_ONE (male self-loop)
    if x == 1 and y == 1:
        return 1   # BLACK_ONE ⊗ BLACK_ONE = BLACK_ONE (female self-loop)
    
    # Cross-gender identity cases (RED_ONE is even/male, BLACK_ONE is odd/female)
    if x == -1 and y == 1:
        return 0  # RED_ONE ⊗ BLACK_ONE: male's female self is undefined
    if x == 1 and y == -1:
        return 0  # BLACK_ONE ⊗ RED_ONE: female's male self is undefined
    
    # Identity on LEFT (lhs): Just a starting point marker, treat as 1 for bit-shift
    # Example: RED_ONE ⊗ 3 = "male-me's mother" = 3
    if x == -1:
        x = 1  # Treat RED_ONE as 1 for composition
    
    # Identity on RIGHT (rhs): Acts as gender/parity filter
    # Example: 3 ⊗ RED_ONE = "mother's male self" = 0 (undefined, mother is female/odd)
    #          3 ⊗ BLACK_ONE = "mother's female self" = 3 (mother is female/odd)
    
    # When y is RED_ONE (-1): Filters for even values only
    if y == -1:
        if x & 1:  # x is odd
            return 0  # Odd values have no male self
        else:
            return x  # Even values' male self is themselves
    
    # When y is BLACK_ONE (1): Filters for odd values only
    if y == 1:
        if x & 1:  # x is odd
            return x  # Odd values' female self is themselves
        else:
            return 0  # Even values have no female self

    # Normal case: bit shifting operation
    bit_position = MSB(y)
    return ((y & (2 ** bit_position - 1)) | (x << bit_position))

## Mathematical Foundations

From here on, $\otimes$ denotes the avos product, $\star$, and $\oplus$ the avos sum, $\dagger$. The carrier is $S = \{0, 1, 2, \dots\}$: $0$ means "no path" and every positive integer encodes a word. The complete development is in `docs/MATHEMATICAL_FOUNDATIONS.md`, and each result is checked against every backend by `tests/avos/test_algebra_properties.py`.

**Lemma 1 (Determinism).** For every vertex $\mathbf{u}$ and every word $w$, at most one path starts at $\mathbf{u}$ and has word $w$.

*Proof.* By induction on the length of $w$. The path for all but the last letter, $k$, ends at a single vertex, which has at most one edge to a vertex of color $k$. $\square$

A Red Black Graph is therefore a partial deterministic automaton over the alphabet $\{\mathsf r, \mathsf b\}$, in which reading the letter $k$ always leads to a vertex of color $k$.

**Lemma 2 (Integer order is shortlex order).** For words $w$ and $w'$, $\iota(w) < \iota(w')$ if and only if $w$ comes before $w'$ in shortlex order.

*Proof.* Words of length $n$ encode to integers in $[2^n, 2^{n+1})$, so shorter words are smaller. Within one length, the integers compare by the bits after the shared leading $1$, which is lexicographic order. $\square$

**Lemma 3 (Compatibility).** If $y$ comes no later than $z$ in shortlex order, then $xy$ comes no later than $xz$, and $yx$ no later than $zx$, for every word $x$.

*Proof.* If $|y| < |z|$, compare lengths. If $|y| = |z|$: in the first case the prefix $x$ is shared; in the second the first difference lies within the first $|y|$ letters. $\square$

### Theorem 1: The Avos Semiring

$(S, \oplus, \otimes, 0, 1)$ is a semiring:

1. $\oplus$ is associative and commutative, with identity $0$. It is also idempotent and selective: $x \oplus y \in \{x, y\}$.
2. $\otimes$ is associative, with $1$ as identity on both sides, and $0$ is absorbing: $0 \otimes x = x \otimes 0 = 0$.
3. $\otimes$ distributes over $\oplus$ on both sides.

Moreover, $1 \oplus x = 1$ for every $x$: the semiring is *bounded*.

*Proof.* (1) $\oplus$ is the minimum for the order in which $0$ is placed above every positive integer. (2) On positive integers, $\otimes$ is concatenation of words carried through the bijection $\iota$, and $1$ encodes the empty word. (3) By Lemmas 2 and 3, $x \otimes \min(y, z) = \min(x \otimes y, x \otimes z)$ and $\min(y, z) \otimes x = \min(y \otimes x, z \otimes x)$. Cases involving $0$ are immediate. Boundedness holds because $1$ is the smallest positive integer. $\square$

This is the semiring of shortlex-minimal words over two letters, a special case of the lexicographic semirings used with weighted automata. It is not commutative ($2 \otimes 3 = 5$ but $3 \otimes 2 = 6$), as expected when composing paths.

### Theorem 2: Closure

Let $A$ be the adjacency matrix without its diagonal: $A_{\mathbf{u}\mathbf{v}} = 2 + c(\mathbf{v})$ if $(\mathbf{u}, \mathbf{v}) \in E$ (2 for an edge to a red vertex, 3 for an edge to a black vertex) and $0$ otherwise. Let $I$ have $1$ on the diagonal and $0$ elsewhere, and use $\oplus$ and $\otimes$ for matrix sums and products. With $n = |V|$:

1. $(A^k)_{\mathbf{u}\mathbf{v}}$ encodes the shortlex-minimal word among paths of length exactly $k$ from $\mathbf{u}$ to $\mathbf{v}$, or is $0$ if there are none.
2. $R^* = I \oplus A \oplus A^2 \oplus \dots \oplus A^{n-1}$ satisfies $R^*_{\mathbf{u}\mathbf{v}} = r(\mathbf{u},\mathbf{v})$.
3. $R^* = I \oplus A R^* = I \oplus R^* A$.
4. $R^* = (I \oplus A)^m$ for every $m \ge n - 1$. In particular, $\lceil \log_2 n \rceil$ squarings of $I \oplus A$ compute $R^*$.

*Proof.* (1) By induction on $k$: a path of length $k$ is a path of length $k-1$ to some vertex followed by one edge, and by Lemma 3 the minimum can be taken prefix by prefix. (2) A DAG has no path with more than $n-1$ edges. (3) Expand both sides using (2). (4) When the sum is idempotent, $(I \oplus A)^m = \bigoplus_{j=0}^{m} A^j$, and $A^j = 0$ for $j \ge n$. $\square$

Because the semiring is bounded, the closure also exists for graphs with cycles: removing a cycle from a path gives a shorter word, so the minimum is always reached by a simple path. A graph is acyclic exactly when the strict closure $A R^*$ is $0$ on the diagonal. Otherwise its diagonal entry at $\mathbf{u}$ encodes the shortest cycle through $\mathbf{u}$.

Theorem 2 justifies each closure algorithm in this library. Floyd–Warshall is correct in any semiring in which the closure exists (Lehmann, 1977). Part 4 justifies closure by repeated squaring, on the CPU and on the GPU. Part 3, read one row at a time in reverse topological order, is the DAG algorithm: the row of $\mathbf{u}$ is $e_\mathbf{u} \oplus \bigoplus_{(\mathbf{u}, \mathbf{v}) \in E} A_{\mathbf{u}\mathbf{v}} \otimes R^*_{\mathbf{v},:}$.

### Theorem 3: Structure of the Closure

For $\mathbf{u} \neq \mathbf{v}$ with $r(\mathbf{u},\mathbf{v}) \neq 0$:

1. **Color.** $r(\mathbf{u},\mathbf{v})$ is even if $\mathbf{v}$ is red and odd if $\mathbf{v}$ is black.
2. **Generation.** $\operatorname{MSB}(r(\mathbf{u},\mathbf{v}))$ is the length of the shortest path from $\mathbf{u}$ to $\mathbf{v}$.
3. **Path.** The bits of $r(\mathbf{u},\mathbf{v})$ after the leading $1$ give the colors along the chosen path, and by Lemma 1 they determine its vertices.
4. **Distinct values.** Each non-zero value appears at most once in a row.
5. **Triangle inequality.** $r(\mathbf{u},\mathbf{w}) \le r(\mathbf{u},\mathbf{v}) \otimes r(\mathbf{v},\mathbf{w})$ whenever the right side is non-zero.

*Proof.* (1) The last bit is the color of the last vertex entered. (2) Lemma 2. (3) The definition of $\iota$. (4) Equal values encode the same word, which by Lemma 1 leads to a single vertex. (5) Joining the two chosen paths gives a path from $\mathbf{u}$ to $\mathbf{w}$, and $r$ is the minimum over all such paths. $\square$

### The ±1 Diagonal

The matrix $R$ used throughout this notebook is $A$ with each vertex's color written on the diagonal: RED_ONE ($-1$) for a red vertex and BLACK_ONE ($1$) for a black one. Write $\sigma_\mathbf{v}$ for this diagonal value. So that matrix products work with this diagonal, the reference implementation above gives the avos product extra rules for these two values:

| Expression | Result | Reading |
| --- | --- | --- |
| $-1 \star x$, $1 \star x$ | $x$ | left identity |
| $x \star -1$ | $x$ if $x$ is even, otherwise $0$ | "$x$'s male self" |
| $x \star 1$ | $x$ if $x$ is odd, otherwise $0$ | "$x$'s female self" |
| $-1 \star -1$, $1 \star 1$ | $-1$, $1$ | a vertex's own diagonal |
| $-1 \star 1$, $1 \star -1$ | $0$ | mixed colors |

**Proposition 4 (Equivalence).** For every Red Black Graph, the closure $R^+$ computed with these rules agrees with $R^*$ off the diagonal, and its diagonal is $\sigma$.

*Proof.* The special values occur only on the diagonal. The left rule is the identity. The right rules are only applied to an entry $R_{\mathbf{u}\mathbf{v}}$ multiplied by $\mathbf{v}$'s own diagonal value, and by Theorem 3 that entry already has the parity of $\mathbf{v}$'s color, so the filter never changes it. The mixed-color rule would need two different colors on one diagonal entry, which cannot happen. $\square$

In tests on random valid graphs, neither rule changed a result in more than 345,000 products. So the $\pm 1$ diagonal is a convenient way to show each vertex's color in the same matrix, and the algebra underneath is the plain product of Theorem 1, where $1$ is the empty path.

Taken as algebra in their own right, the rules above are not associative: $(2 \star 1) \star 3 = 0 \star 3 = 0$, while $2 \star (1 \star 3) = 2 \star 3 = 5$. An earlier version of this notebook concluded from this that the avos algebra is not a semiring and proposed a two-sorted or $\mathbb{Z}/2\mathbb{Z}$-graded structure instead. Theorem 1 shows that this is unnecessary: the non-associativity comes from storing color in the diagonal value, not from the algebra. (The structure is also not $\mathbb{Z}/2\mathbb{Z}$-graded, since the parity of $x \otimes y$ is the parity of $y$, not the sum of the two parities.)

## Transitive Closure for Red-Black Adjacency Matrix

Transitive closure of an adjacency matrix can be computed a number of ways, a simple approach is the [Floyd-Warshall Algorithm](https://en.wikipedia.org/wiki/Floyd%E2%80%93Warshall_algorithm).

Summarized, this algorithm is a tripple loop across the matrix indices continously updating the current transitive relationship, $A^+_{i,j}$, if there is a relationship from $A_{i,k}$ and a relationship from $A_{k,j}$. For $R^+$, the Floyd-Warshall algorithm is modified so that the transitive relationship for $R^+_{i,j}$ is defined as $R^+_{i,j} = R_{i,j} \dagger R_{i,k} \star R_{k,j}$.

A number of matrix operations utilize a "sum of products" pattern (matrix mutliplication, some versions of Floyd-Warshall, etc.). A similar pattern of "avos sum of avos products" is present in operations on matrix representations of Red Black Graphs. 

By Theorem 2 and Proposition 4, this computes $R^+$ for every Red Black Graph.

In [ ]:
# %load ../redblackgraph/reference/transitive_closure.py
from typing import Sequence
from redblackgraph.reference.shortest_path import floyd_warshall
from redblackgraph.types.transitive_closure import TransitiveClosure

def transitive_closure(M: Sequence[Sequence[int]]) -> TransitiveClosure:
    return TransitiveClosure(*floyd_warshall(M))

The example above where $R^+$ was derived by inspection, can now be computed:

In [ ]:
from redblackgraph.reference import transitive_closure
from redblackgraph import RED_ONE, BLACK_ONE

R = [[RED_ONE,  2,  3,  0,  0],
     [ 0, RED_ONE,  0,  2,  0],
     [ 0,  0,  BLACK_ONE,  0,  0],
     [ 0,  0,  0, RED_ONE,  0],
     [ 2,  0,  0,  0,  BLACK_ONE]]
transitive_closure(R).W

## Observations

Given $R^+$, observe that:

* row vectors represent the complete ancestry view for a given vertex
* column vectors represent the complete descendency view for a given vertex. Every non-zero off-diagonal entry in the column of $\mathbf{v}$ has the parity of $\mathbf{v}$'s color (Theorem 3).
* the row vectors of two full siblings agree everywhere except in the siblings' own columns. Half siblings share the ancestry of only one parent.
* the column vectors of two siblings differ once either sibling has descendants represented in the graph
* determining whether $\mathbf{v}$ is an ancestor of $\mathbf{u}$ is $\mathcal{O}(1)$ and provided by $R^+[u,v]$ (with a dense representation of $R^+$)

# Applications of $R^+$

## Calculating Relationship Between Two Rows in $R^+$

With $R^+$ there exists an efficient way to determining full kinship (see: [consanguinity](https://en.wikipedia.org/wiki/Consanguinity)) between any two vertices. 

1. Given two row vectors from $R^+$, $\vec u$ and $\vec v$, consider the common ancestors: the indices $i$ where both $\vec u_{i}$ and $\vec v_{i}$ are non-zero. For each, the generational distances are $d_u = \operatorname{MSB}(\vec u_{i})$ and $d_v = \operatorname{MSB}(\vec v_{i})$.
2. A nearest common ancestor, $\mathbf{i_{min}}$, minimizes $d_u + d_v$.
3. Using a Table of Consanguinity, calculate the relationship from $(d_u, d_v)$.

The simple implementation below minimizes the raw sum $\vec u_{i} + \vec v_{i}$ instead. This usually picks the same ancestor, but not always: an ancestor at distances $(3, 3)$ with values $(8, 8)$ has a smaller raw sum than one at distances $(1, 4)$ with values $(2, 16)$. Pedigree collapse can produce exactly this situation.

**Limits.** $R^+$ keeps one path per pair of vertices: the shortlex-minimal one. On its own it cannot distinguish full siblings (two nearest common ancestors) from half siblings (one) unless all nearest common ancestors are counted, and it cannot compute Wright's coefficient of relationship, which sums over all paths.

### Observation

Determining whether $\mathbf{u}$ is related to $\mathbf{v}$ is $\mathcal{O}(m)$, where $m$ is the number of ancestors recorded for the two vertices (assuming an efficient sparse matrix representation). In a pedigree recorded to $g$ generations, $m \le 2^{g+1} - 2$; in practice it is much smaller, because records thin out in earlier generations and pedigree collapse repeats ancestors.

A simple implementation follows:

In [ ]:
# %load ../redblackgraph/reference/calc_relationship.py
from typing import Sequence
from redblackgraph.types.relationship import Relationship
from redblackgraph.reference import MSB



def cousinth(generational):
    if generational == 2:
        return "1st cousin"
    if generational == 3:
        return "2nd cousin"
    if generational == 4:
        return "3rd cousin"
    return f"{generational - 1}th cousin"

def grandparentith(removal):
    if removal == 4:
        return "2nd"
    if removal == 5:
        return "3rd"
    return f"{removal-2}th"

def lookup_relationship(da: int, db: int) -> str:
    '''
    This is a very rudimentary implementation of a Consanguinity lookup and doesn't handle many
    cases correctly.
    :param da: generational distance from u to common ancestor
    :param db: generational distance from v to common ancestor
    :return: a string designating relationship
    '''
    removal = abs(da - db)
    if da == 0 or db == 0:
        # direct ancestor
        if removal == 1:
            return "parent"
        if removal == 2:
            return "grandparent"
        if removal == 3:
            return "great grandparent"
        return f"{grandparentith(removal)} great grandparent"
    else:
        generational = min(da, db)
        if generational == 1 and removal == 0:
            # sibling
            return "sibling"
        if generational == 1 and removal == 1:
            # parent's sibling
            return "aunt/uncle"
        if removal == 0:
            return cousinth(generational)
        return f"{cousinth(generational)} {removal} removed"


def calculate_relationship(a: Sequence[int], b: Sequence[int]) -> Relationship:
    '''
    Determine if a relationship exists between u, v where u, v are row vectors of the transitive
    closure of a Red Black adjacency matrix
    :param a: row vector for vertex u
    :param b: row vector for vertex v
    :return: (Relationship designation, common ancestor vertex)
    '''

    common_ancestor, (x, y) = min([e for e in enumerate(zip(a, b))
                                   if not e[1][0] == 0 and not e[1][1] == 0],
                                  key=lambda x: x[1][0] + x[1][1],
                                  default=(-1, (0, 0)))

    if common_ancestor == -1:
        return Relationship(-1, "No Relationship")
    return Relationship(common_ancestor,
                        lookup_relationship(MSB(x), MSB(y)))

# Linear Algebra

## Introduction
Having provided a formal definition for a Red Black Graph, looked at its adjacency matrix, $R$, the transitive closure of its adjacency matrix, $R^+$, and the avos sum and product, let's extend these observations into a more general discussion of how principles of linear algebra can be applied to Red Black Graphs.

## Vector Classes
Within the context of a Red Black Graph and its matrix representations, $R$ and $R^+$, the following vector classes are defined:

* *row* vector - represented as $\vec{\mathbf{u}}$. These vectors represent ancestry for a given vertex. Values for elements in these vectors are constrained to whole numbers and -1 where any number, aside from 0, may appear in an alement at most once and where either -1 or 1 must appear as an element but not both. 
* *column* vector - represented by $\vec{\mathbf{v}}$. These vectors represent descendency for a given vertex. Values for elements in these vectors are constrained to whole numbers and -1 where either -1 or 1 must appear as an element but not both. Futhermore if -1 appears as an element any further non-zero integer elements must be even and if 1 appears as an element any further non-zero integer elements must be odd.
* *simple row vector* - represented by $\vec{\mathbf{u}_{s}}$. Row vectors for which elements are constrained to {-1, 0, 1, 2, 3}. These represent a given vertex and it's immediate ancestry only.
* *simple column vector* - represented by $\vec{\mathbf{v}_{s}}$. Column vectors for which elements are constrained to {-1, 0, 1, 2, 3}. These represent a given vertex and it's immediate descendency only.
* *closed row vector* - represented by $\vec{\mathbf{u}_{c}}$. Row vectors from $R^+$. These represent the complete ancestry for a given vertex.
* *closed column vector* - represented by $\vec{\mathbf{v}_{c}}$. Column vectors from $R^+$. These represent the complete descendency for a given vertex.
* *compositional vectors* - represented by $\vec{\mathbf{u}^{c}_{s}}$ or $\vec{\mathbf{v}^{c}_{s}}$. Compositional vectors conform to the constrainst of simple row or column vectors with the following additional constraint: neiter 1 nor -1 appear as an element. The color of the vector is inherent to the vector but not carried as an element. Any consraints due the color are present as if the color were present as an element. Color, if significant, is represented notationally by replacing the supersscript $c$ with the color designation, either $r$ or $b$.


## Avos Product for Vectors

Consider what an avos vector product might represent. Given a row vector and a column vector, the avos product is $r(\mathbf{u},\mathbf{v})$, the relationship between the vertices representing the row and column vectors respectively. 

Consider the $R$ from the transitive closure example:

$$\begin{bmatrix}
-1 & 2 & 3 & 0 & 0 \\
0 & -1 & 0 & 2 & 0 \\
0 & 0 & 1 & 0 & 0 \\
0 & 0 & 0 & -1 & 0 \\
2 & 0 & 0 & 0 & 1 \\
\end{bmatrix}$$

The 4th row vector of $R$ is $\vec{\mathbf{u}_{s}}$ for $vertex_{4}$ while the 2nd column vector is $\vec{\mathbf{v}_{s}}$ for $vertex_{2}$. It is observable by inspection that relationship of $vertex_{4}$ and $vertex_{2}$ is $r(\mathbf{u},\mathbf{v}) == 5$ or:

$$\begin{bmatrix}
2 & 0 & 0 & 0 & 1 \\
\end{bmatrix}
\star
\begin{bmatrix}
3 \\
0 \\
1 \\
0 \\
0 \\
\end{bmatrix}
= 5$$

The vector dot product, summing element-wise products, results in a scaler value of 6. Avos summing of element-wise avos products does yield 5, which represents a relationship.

A simple implementation of the avos vector product follows:

In [ ]:
# %load ../redblackgraph/reference/vec_avos.py
from functools import reduce
from typing import Sequence
from redblackgraph.reference import avos_product, avos_sum

def vec_avos(u: Sequence[int], v: Sequence[int]) -> int:
    '''Given two vectors, compute the avos product.'''
    return reduce( avos_sum, [avos_product(a, b) for a, b in zip(u, v)])

## Observation - Simple/Complete Relationship

The product of a simple row vector and the transitive closure of a Red Black adjacency matrix is a closed row vector
$$\vec{\mathbf{u}_{s}} \star R^+ = \vec{\mathbf{u}_{c}}$$

The product of the transitive closure of a Red Black adjacency matrix and a simple column vector is a closed column vector
$$R^+ \star \vec{\mathbf{v}_{s}} = \vec{\mathbf{v}_{c}}$$

Both follow from Theorem 2. Written one row at a time, $R^* = I \oplus A R^*$ says that the row of $\mathbf{u}$ in $R^*$ is $(e_\mathbf{u} \oplus A_{\mathbf{u},:}) \otimes R^*$, and $e_\mathbf{u} \oplus A_{\mathbf{u},:}$ is exactly the simple row vector of $\mathbf{u}$. (Expanding, $(e_\mathbf{u} \oplus A_{\mathbf{u},:}) \otimes R^* = R^*_{\mathbf{u},:} \oplus A_{\mathbf{u},:} \otimes R^*$, and since part 3 gives $R^*_{\mathbf{u},:} = e_\mathbf{u} \oplus A_{\mathbf{u},:} \otimes R^*$, idempotence of $\oplus$ reduces this to $R^*_{\mathbf{u},:}$.) The column statement is the same argument applied to $R^* = I \oplus R^* A$. With the $\pm 1$ diagonal, the same holds by Proposition 4.

## Avos product for Matrices

With scaler and vector avos products defined, extension to matrices is elementary. Given $A$ and $B$, both matrices following the constraints defined for $R$, and $C = A \star B$, the elements of $C_{ij}$ are given by the vector avos product of $\mathbf{u}_{i}$ from A and $\mathbf{v}_{j}$ from B

Avos matrix multiplication of general matrices seems a little abstract so consider the following practical example. $R \star R$ shows all vertices directly related by following up to 2 relationship edges, $R \star R \star R$ shows all vertices related by following up to 3 relationship edges, etc. By Theorem 2(4), $R$ multiplied by itself $m$ times equals $R^+$ for every $m \ge |V| - 1$, so $\lceil \log_2 |V| \rceil$ repeated squarings of $R$ compute $R^+$.

A simple implementation of the avos matrix product follows:

In [ ]:
# %load ../redblackgraph/reference/mat_avos.py
from functools import reduce
from typing import Sequence
from redblackgraph.reference import avos_product, avos_sum


def mat_avos(A: Sequence[Sequence[int]], B: Sequence[Sequence[int]]) -> Sequence[Sequence[int]]:
    '''Given two matrices, compute the "avos" product.'''
    return [[reduce( avos_sum, [avos_product(a, b) for a, b in zip(A_row, B_col)]) for B_col in zip(*B)] for A_row in A]

## Relational Composition
### Adding a Vertex to $R^+$

Consider the case of adding a new vertex, $\lambda$, to a red black graph. Its edges are described by two compositional vectors (defined above): a row vector, $\vec{\mathbf{u}^{c}_{\lambda,s}}$, whose non-zero entries mark $\lambda$'s parents (2 for a red parent, 3 for a black parent, at most one of each), and a column vector, $\vec{\mathbf{v}^{c}_{\lambda,s}}$, whose non-zero entries mark $\lambda$'s children (each entry is 2 if $\lambda$ is red and 3 if $\lambda$ is black). The color of both vectors is the color of $\lambda$, and no child may already have a parent of $\lambda$'s color. The operation of adding a new vertex to a graph is designated the "vertex relational composition". $R^+$ is a square matrix of dimension $N$, $R_{\lambda}^+$ is a square matrix of dimension $N + 1$, and the notation is:

$$R_{\lambda}^+ = {\vec{\mathbf{u}^{c}_{\lambda,s}} R^+ \vec{\mathbf{v}^{c}_{\lambda,s}}}_{color}$$

By the simple/complete relationship observation, $\vec{\mathbf{u}^{c}_{\lambda,c}} = \vec{\mathbf{u}^{c}_{\lambda,s}} \otimes R^+$ is the complete ancestry of $\lambda$ and $\vec{\mathbf{v}^{c}_{\lambda,c}} = R^+ \otimes \vec{\mathbf{v}^{c}_{\lambda,s}}$ is its complete descendency.

**Theorem 6 (Adding a vertex).** The new graph is acyclic if and only if there is no index $x$ with both $\vec{\mathbf{u}^{c}_{\lambda,c}}_x \neq 0$ and $\vec{\mathbf{v}^{c}_{\lambda,c}}_x \neq 0$. In that case

$$R_\lambda^+ = \begin{pmatrix} R^+ \oplus \vec{\mathbf{v}^{c}_{\lambda,c}} \otimes \vec{\mathbf{u}^{c}_{\lambda,c}} & \vec{\mathbf{v}^{c}_{\lambda,c}} \\ \vec{\mathbf{u}^{c}_{\lambda,c}} & \sigma_\lambda \end{pmatrix},$$

where the outer product is taken entry by entry, $(\vec v \otimes \vec u)_{ij} = \vec v_i \otimes \vec u_j$, and $\sigma_\lambda$ is $-1$ or $1$ according to $\lambda$'s color.

*Proof.* Every new path passes through $\lambda$ exactly once, so it is a path to $\lambda$ followed by a path from $\lambda$. The complete vectors hold the minimal such halves, and by distributivity the update takes the minimum over all combinations. A cycle exists if and only if some vertex is both an ancestor and a descendant of $\lambda$. $\square$

Expressing this algorithmically:

1. generate $\vec{\mathbf{u}^{c}_{\lambda,c}} = \vec{\mathbf{u}^{c}_{\lambda,s}} \otimes R^+$
2. generate $\vec{\mathbf{v}^{c}_{\lambda,c}} = R^+ \otimes \vec{\mathbf{v}^{c}_{\lambda,s}}$
3. if any index is non-zero in both vectors, reject the composition: it would create a cycle
4. compose $R_{\lambda}^+$ by:
    1. appending $\vec{\mathbf{u}^{c}_{\lambda,c}}$ to $R^+$ as a new row
    2. appending $\vec{\mathbf{v}^{c}_{\lambda,c}}$ to $R^+$ as a new column
    3. setting the diagonal element ${R_{\lambda}^+}_{N+1, N+1}$ to either 1 or -1 depending on the color of the composition
    4. for each row, $i$, and each column, $j$, where $\vec{\mathbf{u}^{c}_{\lambda,c}}_{j} \neq 0$, setting ${R_{\lambda}^{+}}_{i,j} = {R^{+}}_{i,j} \oplus (\vec{\mathbf{v}^{c}_{\lambda,c}}_{i} \otimes \vec{\mathbf{u}^{c}_{\lambda,c}}_{j})$

### Adding an Edge to $R^+$

Consider the case of adding a new edge between two existing vertices, $vertex_\alpha$ and $vertex_\beta$, where $vertex_\alpha$ does not already have a parent of $vertex_\beta$'s color. The edge's value is $a = r(\mathbf{\alpha}, \mathbf{\beta})$: 2 if $vertex_\beta$ is red and 3 if it is black. The operation of adding a new edge to a graph is designated the "edge relational composition", with notation:

$$R^+_\lambda = R^+ \star r(\mathbf{\alpha}, \mathbf{\beta})$$

**Theorem 5 (Adding an edge).** The new graph is acyclic if and only if $R^+_{\beta\alpha} = 0$. In that case, for all $i \neq j$,

$${R^+_\lambda}_{ij} = R^+_{ij} \oplus R^+_{i\alpha} \otimes a \otimes R^+_{\beta j},$$

and the diagonal is unchanged. In matrix form, $R^+_\lambda = R^+ \oplus (R^+ e_\alpha) \otimes a \otimes (e_\beta^\top R^+)$, a rank-one update.

*Proof.* A cycle through the new edge needs a path from $\beta$ back to $\alpha$. In an acyclic graph, each new path uses the new edge exactly once, so it is a path from $i$ to $\alpha$, then the new edge, then a path from $\beta$ to $j$. By distributivity and Lemma 3, the minimum over these is the second term. $\square$

Expressing this algorithmically:

1. if $R^+_{\beta\alpha} \neq 0$, reject the composition: it would create a cycle
2. generate the new row for $vertex_\alpha$: $\vec{\mathbf{u}^{c'}_\alpha} = \vec{\mathbf{u}^{'}_\alpha} \otimes R^+$, where $\vec{\mathbf{u}^{'}_\alpha}$ is row $\alpha$ of $R^+$ with element $\beta$ replaced by $a$. This equals $R^+_{\alpha,:} \oplus a \otimes R^+_{\beta,:}$.
3. replace row $\alpha$ of $R^+$ with $\vec{\mathbf{u}^{c'}_\alpha}$
4. for each row, $i \neq \alpha$, and each column, $j$, where $\vec{\mathbf{u}^{c'}_\alpha}_j \neq 0$, set ${R_{\lambda}^{+}}_{i,j} = R^{+}_{i,j} \oplus (R^{+}_{i,\alpha} \otimes \vec{\mathbf{u}^{c'}_\alpha}_j)$

Step 4 agrees with Theorem 5 because $R^{+}_{i,\alpha} \otimes R^{+}_{\alpha,j} \ge R^{+}_{i,j}$ (Theorem 3), so only the new paths through $a$ can lower an entry.

### Simple Implementations

In [ ]:
# %load ../redblackgraph/reference/relational_composition.py
import numpy as np
from redblackgraph.reference.rbg_math import avos_sum, avos_product


def _vertex_compose_pass1(u, A, v):
    N = len(A)
    ul = np.zeros(N, dtype=np.int32)
    vl = np.zeros(N, dtype=np.int32)
    for i in range(N):
        for j in range(N):
            ul[i] = avos_sum(ul[i], avos_product(u[0][j], A[j][i]))
            vl[i] = avos_sum(vl[i], avos_product(A[i][j], v[j][0]))
    return ul, vl


def _vertex_compose_pass2(u, A, v, color):
    N = len(A)
    R = np.zeros((N + 1, N + 1), dtype=np.int32)
    for i in range(N):
        for j in range(N):
            if u[j] != 0:
                R[i, j] = avos_sum(avos_product(v[i], u[j]), A[i][j])
            else:
                R[i, j] = A[i][j]
    for i in range(N):
        R[i, N] = v[i]
        R[N, i] = u[i]
    R[N, N] = color
    return R


def vertex_relational_composition(u, R, v, color):
    '''
    Given simple row vector u, transitively closed matrix R, and simple column vector v where
    u and v represent a vertex, lambda, not currently represented in R, compose R_{lambda}
    which is the transitive closure for the graph with lambda included
    :param u: simple row vector for new vertex, lambda
    :param R: transitive closure for Red Black graph
    :param v: simple column vector for new vertex, lambda
    :param color: color of the node either -1 or 1
    :return: transitive closure of the graph, R, with new node, lambda
    '''
    uc_lambda, vc_lambda = _vertex_compose_pass1(u, R, v)
    return _vertex_compose_pass2(uc_lambda, R, vc_lambda, color)


def _edge_compose_pass1(u, A):
    N = len(A)
    ul = np.zeros(N, dtype=np.int32)
    for i in range(N):
        for j in range(N):
            ul[i] = avos_sum(ul[i], avos_product(u[0][j], A[j][i]))
    return ul


def _edge_compose_pass2(alpha, u, A):
    N = len(A)
    R = [[0] * N for _ in range(N)]
    for i in range(N):
        for j in range(N):
            if i == alpha:
                R[i][j] = u[j]
            elif u[j] != 0:
                R[i][j] = avos_sum(avos_product(A[i][alpha], u[j]), A[i][j])
            else:
                R[i][j] = A[i][j]
    return R


def edge_relational_composition(R, alpha, beta, relationship):
    '''
    Given a transitively closed graph, two vertices in that graph, alpha and beta, and the
    relationship from alpha to beta, compose R', which is the transitive closure with the
    new edge included
    :param R:
    :param alpha: a vertex in the graph (row index)
    :param beta: a vertex in the grpah (column index)
    :param relationship: r(alpha, beta)
    :return: transitive closure of the grpah, R, with new edge
    '''
    N = len(R)
    R[alpha][beta], tmp = relationship, R[alpha][beta]
    u_lambda = _edge_compose_pass1([R[alpha]], R)
    R_lambda = _edge_compose_pass2(alpha, u_lambda, R)
    R[alpha][beta] = tmp
    return R_lambda

# Applications of avos Linear Algebra
## Loop Prevention

An issue that can be encountered in systems that represent familial relationships is the inadvertent injection of graph cycles, resulting in the ["I am my own Grandpa"](https://en.wikipedia.org/wiki/I%27m_My_Own_Grandpa) case. While this is impossible when relationships model sexual reproduction, the introduction of step-relationships, etc. would make this a possibility. Often times there is ambiguity in the available historical records. If a researcher isn't careful, cylces may result as a genealogical model is created. Modifications to both forms of the relational composition algorithms can prevent the introduction of cycles into the graph. 

### Vertex Relational Composition Loop Prevention

As vertices are added to an existing graph via relational composition, the intermediate, complete compositional vectors, $\vec{\mathbf{u}^{c}_{\lambda, c}}$ and $\vec{\mathbf{v}^{c}_{\lambda, c}}$, represent the complete ancestry and complete descendency of the new vertex, $\lambda$. A cycle would require a vertex that appears in both.

By Theorem 6, the **vertex relational composition** is undefined if there exists an index $i$ with $\vec{\mathbf{u}^{c}_{\lambda, c}}_{i} \neq 0$ and $\vec{\mathbf{v}^{c}_{\lambda, c}}_{i} \neq 0$, and is well-formed otherwise.

### Edge Relational Composition Loop Prevention

This case is trivial with a transitively closed matrix. By Theorem 5, given $R^+$ and $r(\mathbf{\alpha},\mathbf{\beta})$, the **edge relational composition** is undefined if $r(\mathbf{\beta},\mathbf{\alpha}) \neq 0$ and well-formed otherwise.

The core (C) implementation performs this check. The reference and sparse implementations do not yet; `tests/avos/test_algebra_properties.py` records this as an expected failure.

## Connected Component Identification
\[**TODO**: rework this section to tie together topological sort + component identification in a single DFS pass. This will be more efficient than current algorithm.\]

As Red Black Graphs are used to represent family relationships, an interesting case is determining how many disjoint trees are represented within a graph: its weakly connected components, where edge direction is ignored. These are usually found with a breadth first or depth first search, or with union-find. (Tarjan's algorithm finds *strongly* connected components, which in a DAG are single vertices.) In a transitively closed adjacency matrix, the row and column of a vertex already list every vertex reachable from it in either direction, so the search is inherently "pre-computed" and the algorithm can simply crawl the matrix.

In [ ]:
# %load ../redblackgraph/reference/components.py
import itertools as it
import numpy as np

from collections import defaultdict
from typing import Dict, Optional, Sequence

def find_components(A: Sequence[Sequence[int]], q:Optional[Dict[int, int]] = None) -> Sequence[int]:
    """
    Given an input adjacency matrix compute the connected components
    :param A: input adjacency matrix (this implementation assumes that it is transitively closed)
    :param q: if set, should be defaultdict(lambda: 0)
    :return: a vector with matching length of A with the elements holding the connected component id of
    the identified connected components
    """

    # Component identification is usually done using iterative dfs for each vertex. Since A is
    # transitively closed, we have implicit DFS info in each row. This algorithm utilizes that
    # fact. Conceptually, this algorithm "crawls" the matrix.
    #
    # This is our algorithm:
    #
    # Allocate an array that will represent the component for each vertex
    # Allocate an array that will represent the vertices that have been visited
    # Iterate over each vertex that hasn't been visited:
    #   This is a new component, so increment the component id
    #   Allocate a set to hold ids to be added to this component
    #   Add the current vertex to this set
    #   while the set is not empty
    #     pull a vextex from the set
    #     add it to the current component
    #     For each non-zero cell in the vertex's row and column add those vertices to the set for this component

    n = len(A)
    if q is None:
        q = defaultdict(lambda: 0)
    vertices = range(n)
    component_for_vertex = np.zeros((n), dtype=np.uint32)
    visited_vertices = np.zeros((n), dtype=np.bool_)
    component_id = 0
    for i in it.filterfalse(lambda x: visited_vertices[x], vertices):
        vertices_added_to_component = set()
        vertex_count = 0
        vertices_added_to_component.add(i)
        while vertices_added_to_component:
            vertex = vertices_added_to_component.pop()
            vertex_count += 1
            visited_vertices[vertex] = True
            component_for_vertex[vertex] = component_id
            for j in vertices:
                if not ((A[vertex][j] == 0 and A[j][vertex] == 0) or visited_vertices[j] or j in vertices_added_to_component):
                    vertices_added_to_component.add(j)
        q[component_id] = vertex_count
        component_id += 1
    return component_for_vertex

Consider the following graph

<img src="img/find-components.png" alt="Graph with Components" style="width: 200px;"/>
<!-- ![Graph with Components](img/find-components.png){ width=50% } -->

By inspection, there are two components and the application of the simplified Tarjan's algorithm identifies which vertices belong to which components.

In [ ]:
from redblackgraph.reference.components import find_components
from redblackgraph import RED_ONE, BLACK_ONE

R = [[RED_ONE, 0, 0, 2, 0, 3, 0],
     [ 0, RED_ONE, 0, 0, 0, 0, 0],
     [ 2, 0, BLACK_ONE, 0, 0, 0, 0],
     [ 0, 0, 0, RED_ONE, 0, 0, 0],
     [ 0, 2, 0, 0, RED_ONE, 0, 3],
     [ 0, 0, 0, 0, 0, BLACK_ONE, 0],
     [ 0, 0, 0, 0, 0, 0, BLACK_ONE]]
find_components(R)

With an efficient sparse representation, this algorithm is linear in the number of non-zero entries of $R^+$, which can be much larger than $|E|$.

## Standard Form

Returning to the example, it is obvious from inspection that one component consists of 4 nodes, the other of 3, and that the diameter of the larger component is 2, while the diameter of the smaller is 1. Observe that symmetrically permuting a matrix corresponds to relabeling the vertices of the associated graph. With an appropriate relabeling, the adjacency matrix of any Red Black Graph is upper triangular: any topological ordering of the vertices will do. A more useful relabeling also groups the vertices of each component together and makes each component's size and diameter easy to read off. The result is $R^{+_c} = \mathbf{P} R^+ \mathbf{P}^\top$, where $\mathbf{P}$ is a permutation matrix.

The implementation calls this the *canonical ordering*. To arrive at $\mathbf{P}$, vertices are sorted by the size of their component (largest first), then component id, then the vertex's total ancestor generations (largest first), then its largest relationship value as an ancestor, then color, and finally vertex id.

Two caveats apply:

* The ordering is not canonical in the graph-theoretic sense. Ties are broken by component id and vertex id, which depend on the original labeling, so two isomorphic graphs can produce different matrices. A true canonical form would be the same for every labeling, and computing one for DAGs is as hard as graph isomorphism. "Standard form" describes what this ordering provides.
* The ordering is not guaranteed to be upper triangular. Total ancestor generations, and largest relationship value, are computed from shortest paths, and under pedigree collapse a child can reach its parent's ancestors by shorter routes than the parent does. For example, if a vertex's parents are $\mathbf{v}$ and $\mathbf{v}$'s great-grandparent, the vertex can be sorted before $\mathbf{v}$. Sorting by *height* (the length of the longest path to an ancestor with no parents) within each component always gives an upper triangular result.

A simple implementation of the standard ordering, and of a plain topological ordering, follows.

In [ ]:
# %load ../redblackgraph/reference/ordering.py
import numpy as np

from collections import defaultdict
from typing import Dict, List, Sequence
from redblackgraph.reference.components import find_components
from redblackgraph.reference.permutation import permute
from redblackgraph.reference.rbg_math import MSB

from redblackgraph.reference.topological_sort import topological_sort
from redblackgraph.types.ordering import Ordering


def _get_permutation(A: Sequence[Sequence[int]], q: Dict[int, int], ids: Sequence[int]) -> List[int]:
    # This is the default sort ordering used by Traingularization
    # it sorts by:
    #   * size of component, descending
    #   * component id, ascending
    #   * relationship count, descending
    #   * max ancestor: ascending
    #   * color: descending
    #   * vertex_id: ascending
    n = len(A)
    max_rel_for_vertex = np.zeros((n), dtype=np.int32)
    ancester_count_for_vertex = np.zeros((n), dtype=np.int32)
    vertices = range(n)
    for i in vertices:
        for j in vertices:
            if A[i][j]:
                ancester_count_for_vertex[i] += MSB(A[i][j])
            if A[j][i]:
                max_rel_for_vertex[i] = max(max_rel_for_vertex[i], A[j][i])

    basis = [i for i in range(len(ids))]
    # sort descending on size of component and "ancestor count", ascending on all other elements
    basis.sort(key=lambda x: (-q[ids[x]], ids[x], -ancester_count_for_vertex[x], max_rel_for_vertex[x], -A[x][x], x))
    return basis


def avos_canonical_ordering(A: Sequence[Sequence[int]]) -> Ordering:
    """
    Canonically sort the matrix.

    This ordering is canonical. Graph components will appear in adjacent
    rows starting with the largest component in rows 0-n, the next largest in n+1-m, etc.
    Should the graph hold components of the same size, the component id will be used to
    order one above the other. Within a component, row ordering is determined first by
    maximum relationship value in a row and finally by original vertex id.

    :param A: input matrix (assumed to be transitively closed)
    :return: an upper triangular matrix that is isomorphic to A
    """

    q = defaultdict(lambda: 0) # dictionary keyed by component id, value is count of vertices in componenet
    components = find_components(A, q)
    perumutation =  _get_permutation(A, q, components)
    return Ordering(permute(A, perumutation), perumutation, q)

def topological_ordering(A: Sequence[Sequence[int]]) -> Ordering:
    """
    Relabel the graph so that the resultant Red Black adjacency matrix is upper triangular

    This ordering is not canonical, it is only guaranteed to produce an upper
    triangular representation. It does so by topologically sorting the graph (O(V+E)).

    Whereas canonical_sort assumes that the input matrix is transitively closed,
    this version does not.
    :param A: input red black adjacency matrix
    :return: an upper triangular matrix that is symmetrical to A (a relabeling of the graph vertices)
    """

    # step 1: determine topological ordering of nodes in the graph
    ordering = topological_sort(A)
    # step 2: permute the matrix and return triangularization
    return Ordering(permute(A, ordering), ordering, dict())

In [ ]:
from redblackgraph.reference import topological_ordering, transitive_closure
topological_ordering(transitive_closure(R).W).A

<img src="img/canonical.png" alt="Graph with Components (Canonical Form)" style="width: 200px;"/>
<!-- ![Graph with Components (Canonical Form)](img/canonical.png){ width=50% } -->


# Appendix A
## Determinants

In ordinary integer arithmetic, $R$ is similar, by a permutation, to an upper triangular matrix whose diagonal is the vertex colors (any topological ordering gives one). The determinant is therefore the product of the diagonal:

$\det R =
    \begin{cases}
            1, &         \text{if } |V_{red}| \text{ is even},\\
            -1, &         \text{if } |V_{red}| \text{ is odd}.
    \end{cases}$

This is a statement about ordinary arithmetic, not the avos operations. The avos semiring has no subtraction, so there is no avos determinant.

# Appendix B
## Implementation Notes on Numpy Extension

Following are some python examples. In addition to the simple implementation presented above in pure python, for performance optimized linear algebra operations, extension modules are provided by Numpy, SciPy, etc. The redblackgraph module also provides extension model implementations that are described below

## rb.array / rb.matrix
The redblackgraph module provides two Numpy extensions, one for array and one for matrix.

The distinctive characteristics of these classes are matrix multiplication has been overridden to support the avos product, as well as methods defined for transitive_closure and relational_composition

To motivate the examples, let's model my familial relationshps. I'm (D) the child of Ewald (E) and Regina (R). Ewald and Marta (M) also have a child, my half-brother, Harald (H). Ewald's parents were Michael (Mi) and Amalie (A). Regenia's parents were John (J) and Inez (I). John also had a son Donald (Do) with Evelyn (Ev). Michael's parents were George (G) and Mariea (Ma). Finally, John's parents were Samuel (S) and Emeline (Em).

This set of relationships is represented by the graph below

<img src="img/small-graph.png" alt="Graph for Exploring Python Implementation" style="width: 300px;"/>
<!-- ![Graph for Exploring Python Implementation](img/small-graph.png){ width=75% } -->

We'll model this as a RedBlackGraph denoting each vertex numerically in the order introduced in the above narrative, e.g. D:0, E:1, R:2, M:3, H:4, Mi:5, A:6, J:7, I:8, Do:9, Ev:10, G:11, Ma:12, S:13, Em14

In these examples, we'll first calculate transitive closure then we'll remove the node (row/column) for John, create the simple row and column vectors for John and use a relational composition to recostruct a transitive closure equivalent. Finally we'll get some timings to compare implementations.

## Simple Implementation
### Transitive Closure

In [ ]:
import numpy as np
import redblackgraph.reference as smp
import copy
#      D   E   R   M   H  Mi   A   J   I  Do  Ev   G  Ma   S  Em
A = [[-1,  2,  3,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0], # D
     [ 0, -1,  0,  0,  0,  2,  3,  0,  0,  0,  0,  0,  0,  0,  0], # E
     [ 0,  0,  1,  0,  0,  0,  0,  2,  3,  0,  0,  0,  0,  0,  0], # R
     [ 0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0], # M
     [ 0,  2,  0,  3, -1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0], # H
     [ 0,  0,  0,  0,  0, -1,  0,  0,  0,  0,  0,  2,  3,  0,  0], # Mi
     [ 0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0], # A
     [ 0,  0,  0,  0,  0,  0,  0, -1,  0,  0,  0,  0,  0,  2,  3], # J
     [ 0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0], # I
     [ 0,  0,  0,  0,  0,  0,  0,  2,  0, -1,  3,  0,  0,  0,  0], # Do
     [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0], # Ev
     [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, -1,  0,  0,  0], # G
     [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0], # Ma
     [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, -1,  0], # S
     [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1]  # Em
    ]
B = copy.deepcopy(A)
res = smp.transitive_closure(B)
print(f"A_star:\n{res.W} \ndiameter: {res.diameter}")

### Vertex Relational Composition
For illustrative purposes, let's remove John from the rb.array representation of the graph

In [ ]:
#       D   E   R   M   H  Mi   A   I  Do  Ev   G  Ma   S  Em
A1 = [[-1,  2,  3,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0], # D
      [ 0, -1,  0,  0,  0,  2,  3,  0,  0,  0,  0,  0,  0,  0], # E
      [ 0,  0,  1,  0,  0,  0,  0,  3,  0,  0,  0,  0,  0,  0], # R
      [ 0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0], # M
      [ 0,  2,  0,  3, -1,  0,  0,  0,  0,  0,  0,  0,  0,  0], # H
      [ 0,  0,  0,  0,  0, -1,  0,  0,  0,  0,  2,  3,  0,  0], # Mi
      [ 0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0], # A
      [ 0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0], # I
      [ 0,  0,  0,  0,  0,  0,  0,  0, -1,  3,  0,  0,  0,  0], # Do
      [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0], # Ev
      [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0, -1,  0,  0,  0], # G
      [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0], # Ma
      [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, -1,  0], # S
      [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1]  # Em
     ]
B1 = copy.deepcopy(A1)
res = smp.transitive_closure(B1)
print(f"A1_star:\n{res.W} \ndiameter: {res.diameter}")

**Observation**: I am no longer related to Samuel nor Emeline, but that the diameter is still 3 (my relationship to George and Mariea).

Let's look at the row (u) and column (v) vectors that would define John in relationship to A1 as well as the relational_composition of A1 with u and v.

In [ ]:
#      D   E   R   M   H  Mi   A   I  Do  Ev   G  Ma   S  Em
u = [[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  2,  3]] 
v = [[ 0], # D  
     [ 0], # E
     [ 2], # R
     [ 0], # M
     [ 0], # H
     [ 0], # Mi
     [ 0], # A
     [ 0], # I
     [ 2], # Do
     [ 0], # Ev
     [ 0], # G
     [ 0], # Ma
     [ 0], # S
     [ 0], # Em
    ]
A_lambda = smp.vertex_relational_composition(u, A1, v, -1)
A_lambda

### Edge Transitive Closure
Using the above example, remove the relationship from Regina to John

In [ ]:
#        D   E   R   M   H  Mi   A   I  Do  Ev   G  Ma   S  Em   J
R1 = [[ -1,  2,  3,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],  # D
      [  0, -1,  0,  0,  0,  2,  3,  0,  0,  0,  0,  0,  0,  0,  0],  # E
      [  0,  0,  1,  0,  0,  0,  0,  3,  0,  0,  0,  0,  0,  0,  0],  # R
      [  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],  # M
      [  0,  2,  0,  3, -1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],  # H
      [  0,  0,  0,  0,  0, -1,  0,  0,  0,  0,  2,  3,  0,  0,  0],  # Mi
      [  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0],  # A
      [  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0],  # I
      [  0,  0,  0,  0,  0,  0,  0,  0, -1,  3,  0,  0,  0,  0,  2],  # Do
      [  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0],  # Ev
      [  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, -1,  0,  0,  0,  0],  # G
      [  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0],  # Ma
      [  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, -1,  0,  0],  # S
      [  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0], # Em
      [  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  2,  3, -1]  # J
      ]
R = smp.transitive_closure(R1).W
# Missing edge is R -> J, 2
A_lambda = smp.edge_relational_composition(R, 2, 14, 2)
A_lambda

### Timings

In [ ]:
%%timeit
B1 = copy.deepcopy(A1)
res = smp.transitive_closure(B1)

In [ ]:
%%timeit
A_lambda = smp.vertex_relational_composition(u, A1, v, -1)

In [ ]:
%%timeit
A_lambda = smp.edge_relational_composition(R, 2, 14, 2)

## Optimized Implementation
### Transitive Closure

In [ ]:
import redblackgraph as rb
#               D   E   R   M   H  Mi   A   J   I  Do  Ev   G  Ma   S  Em
A = rb.array([[-1,  2,  3,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0], # D
              [ 0, -1,  0,  0,  0,  2,  3,  0,  0,  0,  0,  0,  0,  0,  0], # E
              [ 0,  0,  1,  0,  0,  0,  0,  2,  3,  0,  0,  0,  0,  0,  0], # R
              [ 0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0], # M
              [ 0,  2,  0,  3, -1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0], # H
              [ 0,  0,  0,  0,  0, -1,  0,  0,  0,  0,  0,  2,  3,  0,  0], # Mi
              [ 0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0], # A
              [ 0,  0,  0,  0,  0,  0,  0, -1,  0,  0,  0,  0,  0,  2,  3], # J
              [ 0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0], # I
              [ 0,  0,  0,  0,  0,  0,  0,  2,  0, -1,  3,  0,  0,  0,  0], # Do
              [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0], # Ev
              [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, -1,  0,  0,  0], # G
              [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0], # Ma
              [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, -1,  0], # S
              [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1]  # Em
             ], dtype=np.int32)

In [ ]:
result = A.transitive_closure()
print(f"A_star:\n{result.W} \ndiameter: {result.diameter}")

### Vertex Relational Composition
For illustrative purposes, let's remove John from the rb.array representation of the graph

In [ ]:
#                D   E   R   M   H  Mi   A   I  Do  Ev   G  Ma   S  Em
A1 = rb.array([[-1,  2,  3,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0], # D
               [ 0, -1,  0,  0,  0,  2,  3,  0,  0,  0,  0,  0,  0,  0], # E
               [ 0,  0,  1,  0,  0,  0,  0,  3,  0,  0,  0,  0,  0,  0], # R
               [ 0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0], # M
               [ 0,  2,  0,  3, -1,  0,  0,  0,  0,  0,  0,  0,  0,  0], # H
               [ 0,  0,  0,  0,  0, -1,  0,  0,  0,  0,  2,  3,  0,  0], # Mi
               [ 0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0], # A
               [ 0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0], # I
               [ 0,  0,  0,  0,  0,  0,  0,  0, -1,  3,  0,  0,  0,  0], # Do
               [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0], # Ev
               [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0, -1,  0,  0,  0], # G
               [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0], # Ma
               [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, -1,  0], # S
               [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1]  # Em
              ], dtype=np.int32)
result = A1.transitive_closure()
print(f"A1_star:\n{result.W} \ndiameter: {result.diameter}")
A1_star = result.W

**Observation**: I am no longer related to Samuel nor Emeline, but that the diameter is still 3 (my relationship to George and Mariea).

Let's look at the row (u) and column (v) vectors t

In [ ]:
#               D   E   R   M   H  Mi   A   I  Do  Ev   G  Ma   S  Em
u = rb.array([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  2,  3]], dtype=np.int32) 
v = rb.array([[ 0],  
              [ 0],
              [ 2],
              [ 0],
              [ 0],
              [ 0],
              [ 0],
              [ 0],
              [ 2],
              [ 0],
              [ 0],
              [ 0],
              [ 0],
              [ 0],
             ], dtype=np.int32) 

u_lambda = u @ A1_star
v_lambda = A1_star @ v
print(f"u_lambda:\n{u_lambda}")
print(f"v_lambda:\n{v_lambda}")

A_lambda = A1_star.vertex_relational_composition(u, v, -1)
print(f"A_lambda:\n{A_lambda}")

### Edge Transitive Closure
Using the above example, remove the relationship from Regina to John

In [ ]:
#                 D   E   R   M   H  Mi   A   I  Do  Ev   G  Ma   S  Em   J
R1 = rb.array([[ -1,  2,  3,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],  # D
               [  0, -1,  0,  0,  0,  2,  3,  0,  0,  0,  0,  0,  0,  0,  0],  # E
               [  0,  0,  1,  0,  0,  0,  0,  3,  0,  0,  0,  0,  0,  0,  0],  # R
               [  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],  # M
               [  0,  2,  0,  3, -1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],  # H
               [  0,  0,  0,  0,  0, -1,  0,  0,  0,  0,  2,  3,  0,  0,  0],  # Mi
               [  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0,  0],  # A
               [  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0,  0,  0],  # I
               [  0,  0,  0,  0,  0,  0,  0,  0, -1,  3,  0,  0,  0,  0,  2],  # Do
               [  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,  0],  # Ev
               [  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, -1,  0,  0,  0,  0],  # G
               [  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0],  # Ma
               [  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, -1,  0,  0],  # S
               [  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0],  # Em
               [  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  2,  3, -1]   # J
      ])
R = R1.transitive_closure().W
# Missing edge is R -> J, 2
A_lambda = R.edge_relational_composition(2, 14, 2)
A_lambda

### Timings

In [ ]:
%%timeit
result = A.transitive_closure()
A1_star = result.W

In [ ]:
%%timeit
A_lambda = A1_star.vertex_relational_composition(u, v, -1)

In [ ]:
%%timeit
A_lambda = R.edge_relational_composition(2, 14, 2)

## Miscellaneous Linear Algebra

In [ ]:
from numpy.linalg import det
det(A_lambda)

In [ ]:
A_lambda.cardinality()

# Appendix C
## Related Work

The Red-Black Graph (RBG) formalism—directed acyclic graphs with per-vertex colors and the restriction of at most one outgoing edge per color—intersects with several established areas of graph theory and applied data structures.

**Ahnentafel numbering.**  
The relationship value $r(\mathbf{u},\mathbf{v})$ is the Ahnentafel number of $\mathbf{v}$ in $\mathbf{u}$'s pedigree, the genealogical numbering in which self is 1, the father of $n$ is $2n$ and the mother is $2n+1$ (Eytzinger, 1590; Sosa, 1676; Kekulé von Stradonitz, 1898). The same numbering is the Eytzinger layout of an implicit binary tree in computer science.

**Algebraic path problems and lexicographic semirings.**  
Theorem 1 places the avos algebra in the theory of semirings for path problems (Carré, 1971; Lehmann, 1977; Gondran & Minoux, 2008; Mohri, 2002), which supplies the correctness of Floyd–Warshall, repeated squaring and the incremental updates. The avos semiring itself is the shortlex-minimal word semiring over two letters, a special case of the lexicographic semirings used for weighted automata (Roark, Sproat & Shafran, 2011).

**Multitrees.**  
Multitrees are DAGs in which the reachable subgraph from any node is a tree, or equivalently, there exists at most one directed path between any pair of vertices. They have been studied as a way to capture overlapping hierarchies while avoiding cycles. Like RBGs, multitrees are motivated by the need to embed multiple trees into a single DAG. Unlike multitrees, RBGs allow several paths between two vertices (pedigree collapse), and multitrees do not enforce per-color outdegree constraints ([definition on Wikipedia](https://en.wikipedia.org/wiki/Multitree)).

**Multiple-tree DAG representations.**  
The information visualization community has explored representing multiple classification trees within a single DAG to expose overlaps and differences. For example, Graham and Kennedy’s *Exploring Multiple Trees through DAG Representations* (IEEE TVCG, 2007) develops techniques to merge and visualize multiple tree structures simultaneously ([DOI link](https://doi.org/10.1109/TVCG.2007.70536)). This closely parallels the “interleaved binary trees” intuition behind RBGs.

**Polyhierarchies and ontologies.**  
Applied ontologies such as [SNOMED CT](https://www.snomed.org/) are routinely modeled as DAGs to accommodate multiple inheritance, where a concept can belong to several hierarchies simultaneously. These “polyhierarchies” resonate with the RBG aim of combining multiple overlapping tree structures into a unified directed acyclic form.

**Automata and color-constrained graphs.**  
In automata theory, deterministic finite automata ([DFA](https://en.wikipedia.org/wiki/Deterministic_finite_automaton)) impose the invariant that from each state there is at most one outgoing edge per input symbol. This is more than an analogy: by Lemma 1, an RBG is a partial deterministic automaton over the alphabet {red, black}, and $r(\mathbf{u},\mathbf{v})$ encodes the shortlex-least word leading from $\mathbf{u}$ to $\mathbf{v}$. When minimized, such automata become DAGs (DAFSAs) representing large sets of strings compactly. Similarly, work on [edge-colored digraphs](https://en.wikipedia.org/wiki/Edge_coloring) and alternating or properly colored paths addresses reachability under color constraints, though without the structural tree embedding that RBGs enforce.

**Shared substructure DAGs.**  
Several canonical data structures demonstrate the compression of multiple trees into DAGs. Directed Acyclic Word Graphs ([DAWGs](https://en.wikipedia.org/wiki/Deterministic_acyclic_finite_state_automaton)) merge tries by sharing suffixes, producing a minimal DAG that encodes many overlapping word trees. Binary Decision Diagrams ([BDDs](https://en.wikipedia.org/wiki/Binary_decision_diagram)) and their multi-valued generalizations collapse isomorphic subtrees of decision trees into compact DAGs. These approaches are structurally similar to the RBG motivation of compactly encoding multiple interleaved trees.

**Domain-specific analogues.**  
In population genetics, ancestral recombination graphs ([ARGs](https://en.wikipedia.org/wiki/Ancestral_recombination_graph)) model how a sequence of local genealogical trees are joined into a global DAG through recombination events. While the semantics differ, ARGs exemplify how multiple tree topologies can coexist within a single DAG framework.

---

**Summary.**  
The Red-Black Graph lies at the intersection of multitree theory, color-constrained digraphs, and DAG-based compression of tree families. Its unique contribution is combining (a) interleaving of multiple binary trees, (b) per-vertex color assignments, and (c) a constraint of at most one outgoing edge per color, together with algebraic tools for analyzing relationships via adjacency and closure matrices. This combination does not appear in prior graph-theoretic literature, though it is thematically aligned with several strands of work in automata, ontologies, and data structure compression.


## References

* Carré, B. A. (1971). An algebra for network routing problems. *IMA Journal of Applied Mathematics*, 7(3).
* Gondran, M., & Minoux, M. (2008). *Graphs, Dioids and Semirings*. Springer.
* Lehmann, D. J. (1977). Algebraic structures for transitive closure. *Theoretical Computer Science*, 4(1).
* Mohri, M. (2002). Semiring frameworks and algorithms for shortest-distance problems. *Journal of Automata, Languages and Combinatorics*, 7(3).
* Roark, B., Sproat, R., & Shafran, I. (2011). Lexicographic semirings for exact automata encoding of sequence models. *Proceedings of ACL*.
* Wright, S. (1922). Coefficients of inbreeding and relationship. *The American Naturalist*, 56.
* Ahnentafel numbering: Eytzinger (1590); Sosa (1676); Kekulé von Stradonitz (1898).